# Robot Path Finding using AI Search Algorithms

**Name:** Harshini Ramdas Bhandary  
**Division:** A  
**Batch:** A2  
**PRN:** 202401040151  
**Roll No:** 35

**Title:** Performance evaluation of problem-solving search algorithms – Uninformed, Informed, Local Search and Constraint Satisfaction.

**Aim:** To implement and compare BFS, DFS, DLS, IDDFS, A*, Hill Climbing, Genetic Algorithm and CSP Backtracking on a robot path-finding problem in a 4 × 4 grid.

In [1]:
from collections import deque
import heapq
import random
import time

## Robot Path Finding – Comparison of AI Search Algorithms

In [2]:
# ============================================================
# ROBOT PATH FINDING - COMPARISON OF AI SEARCH ALGORITHMS
# ============================================================

# Grid legend:
# 0 -> free cell
# 1 -> obstacle (blocked cell)

grid = [
    [0, 0, 0, 1],
    [0, 1, 0, 0],
    [0, 0, 0, 1],
    [0, 1, 0, 0]
]

start = (0, 0)   # initial state of the robot
goal = (3, 3)    # goal state of the robot

ROWS = len(grid)
COLS = len(grid[0])

## State Space (all free cells the robot can occupy)

In [3]:
# ============================================================
# STATE SPACE (all free cells the robot can occupy)
# ============================================================

state_space = [
    (r, c)
    for r in range(ROWS)
    for c in range(COLS)
    if grid[r][c] == 0
]

print("STATE SPACE:")
print(state_space)

STATE SPACE:
[(0, 0), (0, 1), (0, 2), (1, 0), (1, 2), (1, 3), (2, 0), (2, 1), (2, 2), (3, 0), (3, 2), (3, 3)]


## Actions (Up, Down, Left, Right)

In [4]:
# ============================================================
# ACTIONS (Up, Down, Left, Right)
# ============================================================

moves = [
    (-1, 0),   # Up
    (1, 0),    # Down
    (0, -1),   # Left
    (0, 1)     # Right
]

## Successor Function

In [5]:
# ============================================================
# SUCCESSOR FUNCTION
# ============================================================

def get_neighbors(state):
    neighbors = []
    for dr, dc in moves:
        nxt = (state[0] + dr, state[1] + dc)
        if nxt in state_space:
            neighbors.append(nxt)
    return neighbors

## Build Path From Parent Map

In [6]:
# ============================================================
# BUILD PATH FROM PARENT MAP
# ============================================================

def build_path(parent):
    path = []
    node = goal
    while node != start:
        path.append(node)
        node = parent[node]
    path.append(start)
    path.reverse()
    return path

## Heuristic – Manhattan Distance to Goal

In [7]:
# ============================================================
# HEURISTIC - MANHATTAN DISTANCE TO GOAL
# ============================================================

def manhattan(state):
    return abs(state[0] - goal[0]) + abs(state[1] - goal[1])

## 1. Breadth-First Search (Uninformed)

In [8]:
# ============================================================
# 1. BREADTH-FIRST SEARCH (Uninformed)
# ============================================================

def bfs():
    frontier = deque([start])
    visited = {start}
    parent = {}
    explored = 0

    while frontier:
        node = frontier.popleft()
        explored += 1

        if node == goal:
            return build_path(parent), explored

        for nxt in get_neighbors(node):
            if nxt not in visited:
                visited.add(nxt)
                parent[nxt] = node
                frontier.append(nxt)

    return None, explored

## 2. Depth-First Search (Uninformed)

In [9]:
# ============================================================
# 2. DEPTH-FIRST SEARCH (Uninformed)
# ============================================================

def dfs():
    frontier = [start]
    visited = {start}
    parent = {}
    explored = 0

    while frontier:
        node = frontier.pop()
        explored += 1

        if node == goal:
            return build_path(parent), explored

        for nxt in get_neighbors(node):
            if nxt not in visited:
                visited.add(nxt)
                parent[nxt] = node
                frontier.append(nxt)

    return None, explored

## 3. Depth-Limited Search (Uninformed)

In [10]:
# ============================================================
# 3. DEPTH-LIMITED SEARCH (Uninformed)
# ============================================================

DEPTH_LIMIT = 6


def dls(limit=DEPTH_LIMIT):
    frontier = [(start, 0, [start])]       # (state, depth, path so far)
    explored = 0

    while frontier:
        node, depth, path = frontier.pop()
        explored += 1

        if node == goal:
            return path, explored

        if depth < limit:                  # cutoff: do not expand at depth L
            for nxt in get_neighbors(node):
                if nxt not in path:        # avoid cycles on the current path
                    frontier.append((nxt, depth + 1, path + [nxt]))

    return None, explored

## 4. Iterative Deepening DFS (Uninformed)

In [11]:
# ============================================================
# 4. ITERATIVE DEEPENING DFS (Uninformed)
# ============================================================

def iddfs(max_depth=len(state_space)):
    total_explored = 0

    for limit in range(max_depth + 1):     # limit = 0, 1, 2, ...
        path, explored = dls(limit)
        total_explored += explored
        if path is not None:
            return path, total_explored

    return None, total_explored

## 5. A* Search (Informed): f(n) = g(n) + h(n)

In [12]:
# ============================================================
# 5. A* SEARCH (Informed)  f(n) = g(n) + h(n)
# ============================================================

def a_star():
    frontier = [(manhattan(start), 0, start)]   # (f, g, state)
    visited = set()
    parent = {}
    explored = 0

    while frontier:
        f, g, node = heapq.heappop(frontier)

        if node in visited:
            continue
        visited.add(node)
        explored += 1

        if node == goal:
            return build_path(parent), explored

        for nxt in get_neighbors(node):
            if nxt not in visited:
                new_g = g + 1
                parent[nxt] = node
                heapq.heappush(frontier, (new_g + manhattan(nxt), new_g, nxt))

    return None, explored

## 6. Hill Climbing (Local Search)

In [13]:
# ============================================================
# 6. HILL CLIMBING (Local Search)
# ============================================================

def hill_climbing():
    node = start
    path = [node]
    visited = {node}
    explored = 1

    while node != goal:
        candidates = [s for s in get_neighbors(node) if s not in visited]

        if not candidates:                 # dead end
            return None, explored

        best = min(candidates, key=manhattan)

        # No neighbour is better -> stuck at a local optimum
        if manhattan(best) >= manhattan(node):
            return None, explored

        node = best
        visited.add(node)
        path.append(node)
        explored += 1

    return path, explored

## 7. Genetic Algorithm (Local Search / Optimization)

In [14]:
# ============================================================
# 7. GENETIC ALGORITHM (Local Search / Optimization)
# ============================================================
# A chromosome is a list of genes, each gene is a move:
# 0 = Up, 1 = Down, 2 = Left, 3 = Right

def decode(chromosome):
    """Convert a chromosome into a path and check the constraints."""
    node = start
    path = [node]

    for gene in chromosome:
        dr, dc = moves[gene]
        r, c = node[0] + dr, node[1] + dc

        # Constraint 1: robot must stay inside the grid
        if not (0 <= r < ROWS and 0 <= c < COLS):
            return path, False

        # Constraint 2: robot cannot enter an obstacle
        if grid[r][c] == 1:
            return path, False

        node = (r, c)
        path.append(node)

    return path, True


def fitness(chromosome):
    path, valid = decode(chromosome)
    last = path[-1]
    dist = manhattan(last)

    if last == goal:                       # goal reached: shorter is better
        return 1000 + (100 - len(path))
    if valid:                              # valid but not at goal
        return 100 - dist * 10 - len(path)
    return 10 - dist * 20                  # constraint violated: penalty


def random_chromosome(length):
    return [random.randint(0, 3) for _ in range(length)]


def initial_population(size, length):
    return [random_chromosome(length) for _ in range(size)]


def select(population):
    population.sort(key=fitness, reverse=True)
    return population[:10]                 # keep the 10 fittest


def crossover(p1, p2):
    cut = random.randint(1, len(p1) - 1)
    return p1[:cut] + p2[cut:], p2[:cut] + p1[cut:]


def mutate(chromosome, rate=0.1):
    for i in range(len(chromosome)):
        if random.random() < rate:
            chromosome[i] = random.randint(0, 3)
    return chromosome


def genetic_algorithm():
    random.seed(42)                        # reproducible results

    POP_SIZE = 100
    CHROM_LEN = 12
    GENERATIONS = 200

    population = initial_population(POP_SIZE, CHROM_LEN)
    explored = 0
    best, best_fit = None, float("-inf")

    for _ in range(GENERATIONS):
        for chrom in population:
            fit = fitness(chrom)
            explored += 1
            if fit > best_fit:
                best_fit, best = fit, chrom[:]

        path, valid = decode(best)
        if path[-1] == goal and valid:
            return path, explored

        elite = select(population)
        next_gen = [elite[0][:]]           # elitism: carry the best forward

        while len(next_gen) < POP_SIZE:
            c1, c2 = crossover(random.choice(elite), random.choice(elite))
            next_gen.append(mutate(c1))
            if len(next_gen) < POP_SIZE:
                next_gen.append(mutate(c2))

        population = next_gen

    return None, explored

## 8. Constraint Satisfaction – Backtracking Search

In [15]:
# ============================================================
# 8. CONSTRAINT SATISFACTION - BACKTRACKING SEARCH
# ============================================================
# Variables : X0, X1, ..., Xk  (cell occupied at step i)
# Domain    : every free cell in the state space
# Constraints:
#   C1: X0 = start and Xk = goal
#   C2: X(i+1) must be adjacent to X(i)      (one legal move)
#   C3: all Xi are different                 (no cell visited twice)
#   C4: goal must still be reachable in the remaining steps
#       (Manhattan distance <= steps left)   -> forward checking
# k is increased from 1 upwards, so the first solution is the shortest.

def csp_backtracking():
    explored = 0

    def consistent(value, assignment, step, k):
        if value in assignment:                         # C3
            return False
        if manhattan(value) > k - step:                 # C4
            return False
        if step == k and value != goal:                 # C1 (last variable)
            return False
        return True

    def backtrack(assignment, k):
        nonlocal explored
        step = len(assignment)
        if step == k + 1:
            return assignment

        for value in get_neighbors(assignment[-1]):     # C2
            explored += 1
            if consistent(value, assignment, step, k):
                result = backtrack(assignment + [value], k)
                if result is not None:
                    return result
        return None                                     # backtrack

    for k in range(1, len(state_space)):
        solution = backtrack([start], k)               # C1: X0 = start
        if solution is not None:
            return solution, explored

    return None, explored

## Performance Evaluation

In [16]:
# ============================================================
# PERFORMANCE EVALUATION
# ============================================================

def evaluate(name, algorithm):
    t0 = time.perf_counter()
    path, explored = algorithm()
    elapsed = (time.perf_counter() - t0) * 1000

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    if path is not None:
        length = len(path) - 1
        print("Path           :", path)
        print("Path Length    :", length)
    else:
        length = "No Path"
        print("No path found")

    print("Nodes Explored :", explored)
    print("Execution Time :", round(elapsed, 6), "ms")

    return [name, length, explored, round(elapsed, 6)]


algorithms = [
    ("BFS - Uninformed Search", bfs),
    ("DFS - Uninformed Search", dfs),
    ("DLS - Uninformed Search", dls),
    ("IDDFS - Uninformed Search", iddfs),
    ("A* - Informed Search", a_star),
    ("Hill Climbing - Local Search", hill_climbing),
    ("Genetic Algorithm - Local Search", genetic_algorithm),
    ("Backtracking - Constraint Satisfaction", csp_backtracking),
]

results = [evaluate(name, fn) for name, fn in algorithms]


BFS - Uninformed Search
Path           : [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (3, 2), (3, 3)]
Path Length    : 6
Nodes Explored : 12
Execution Time : 0.021167 ms

DFS - Uninformed Search
Path           : [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (3, 2), (3, 3)]
Path Length    : 6
Nodes Explored : 11
Execution Time : 0.011958 ms

DLS - Uninformed Search
Path           : [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (3, 2), (3, 3)]
Path Length    : 6
Nodes Explored : 10
Execution Time : 0.010459 ms

IDDFS - Uninformed Search
Path           : [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (3, 2), (3, 3)]
Path Length    : 6
Nodes Explored : 53
Execution Time : 0.034917 ms

A* - Informed Search
Path           : [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (3, 2), (3, 3)]
Path Length    : 6
Nodes Explored : 12
Execution Time : 0.017209 ms

Hill Climbing - Local Search
No path found
Nodes Explored : 4
Execution Time : 0.009125 ms

Genetic Algorithm - Local Search
No path found
Nodes Explored : 20000
Exe

## Final Performance Comparison

In [17]:
# ============================================================
# FINAL PERFORMANCE COMPARISON
# ============================================================

print("\n\n" + "=" * 90)
print("FINAL PERFORMANCE COMPARISON")
print("=" * 90)
print(f"{'Algorithm':<40}{'Path Length':<15}{'Nodes Explored':<20}{'Time (ms)':<15}")
print("-" * 90)
for name, length, explored, t in results:
    print(f"{name:<40}{str(length):<15}{explored:<20}{t:<15}")
print("=" * 90)



FINAL PERFORMANCE COMPARISON
Algorithm                               Path Length    Nodes Explored      Time (ms)      
------------------------------------------------------------------------------------------
BFS - Uninformed Search                 6              12                  0.021167       
DFS - Uninformed Search                 6              11                  0.011958       
DLS - Uninformed Search                 6              10                  0.010459       
IDDFS - Uninformed Search               6              53                  0.034917       
A* - Informed Search                    6              12                  0.017209       
Hill Climbing - Local Search            No Path        4                   0.009125       
Genetic Algorithm - Local Search        No Path        20000               35.369375      
Backtracking - Constraint Satisfaction  6              23                  0.017042       
